# 05 — Disruption Prediction Model
**Module E**

Champion-challenger ensemble for 30-day stockout risk prediction:
- 20+ features spanning graph, detection, forecasting, and inventory layers
- XGBoost champion + LightGBM challenger
- Stacked logistic regression meta-learner
- SHAP TreeExplainer for feature attribution
- Metrics: PR-AUC, Precision@K, prediction lead time

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src.features import build_feature_matrix, split_chronological
from src.models import (
    train_xgboost, train_lightgbm, build_ensemble, compute_shap_values
)
from src.viz import create_shap_summary, create_confusion_matrix_chart

## Build Feature Matrix

In [2]:
feature_matrix = build_feature_matrix()
print(f'Feature matrix shape: {feature_matrix.shape}')
feature_matrix.head()


  ── Building Feature Matrix ──
    Computing target variable (stockout_risk_30d)...
    Adding graph features...
    Adding disruption detection features...
    Adding forecasting features...


    Adding inventory features...
    Encoding categorical features...

    Feature matrix shape: (78000, 28)
    Feature count: 20
    Positive class rate: 12.1%
Feature matrix shape: (78000, 28)


,week_start_date,sku_id,week_num,category,supplier_country,demand_units,stockout_flag,betweenness_centrality,pagerank,degree_centrality,...,forecast_uncertainty_width,demand_trend_slope,current_inventory_weeks_of_cover,days_to_reorder_point,safety_stock_adequacy_ratio,lead_time_deviation_from_normal,sku_category_encoded,supplier_country_encoded,unit_cost_usd,stockout_next_4w
0,2022-01-03,SKU-0001,1,Electronics,Germany,125,False,0.029963,0.010406,0.109192,...,0.000000,0.0,2.560000,17.920000,0.640000,-0.200000,0,0,120.2,0
1,2022-01-10,SKU-0001,2,Electronics,Germany,113,False,0.076057,0.030561,0.124721,...,0.071305,0.0,2.689076,18.823529,0.672269,0.066667,0,0,120.2,0
2,2022-01-17,SKU-0001,3,Electronics,Germany,116,False,0.058560,0.007477,0.087411,...,0.052924,-4.5,2.711864,18.983051,0.677966,-0.133333,0,0,120.2,0
3,2022-01-24,SKU-0001,4,Electronics,Germany,115,False,0.047893,0.031478,0.108698,...,0.045331,-2.7,2.729211,19.104478,0.682303,-0.133333,0,0,120.2,0
4,2022-01-31,SKU-0001,5,Electronics,Germany,146,False,0.012481,0.020411,0.035505,...,0.128296,4.4,2.612245,18.285714,0.653061,0.000000,0,0,120.2,0


In [3]:
X_train, y_train, X_val, y_val, X_test, y_test, feature_names = split_chronological(feature_matrix)


    Train: 39,000 samples (pos rate: 8.5%)
    Val:   6,500 samples (pos rate: 11.5%)
    Test:  32,500 samples (pos rate: 16.5%)


## Train Models

In [4]:
xgb_results = train_xgboost(X_train, y_train, X_val, y_val, feature_names)


  ── Training XGBoost Champion ──


    Train PR-AUC: 0.8756
    Val PR-AUC:   0.1181
    Best iteration: N/A
    Top 5 features:
      disruption_score_current: 0.1872
      cusum_flag_rolling_4w: 0.1248
      sku_category_encoded: 0.0534
      safety_stock_adequacy_ratio: 0.0509
      supplier_country_risk_tier: 0.0463


In [5]:
lgb_results = train_lightgbm(X_train, y_train, X_val, y_val, feature_names)


  ── Training LightGBM Challenger ──


    Train PR-AUC: 0.8299
    Val PR-AUC:   0.1194
    Top 5 features:
      unit_cost_usd: 2594.0000
      current_inventory_weeks_of_cover: 2461.0000
      demand_trend_slope: 2036.0000
      demand_rolling_8w: 1549.0000
      demand_rolling_4w: 1427.0000


## Ensemble

In [6]:
ensemble = build_ensemble(xgb_results, lgb_results, X_val, y_val, X_test, y_test)


  ── Building Ensemble ──
    Weighted Ensemble Val PR-AUC: 0.1188
    Stacked Ensemble Val PR-AUC:  0.1193
    Best ensemble: stacked (PR-AUC: 0.1193)

    Test PR-AUC: 0.2933
    Precision@10%: 0.3637
    Recall@0.3: 0.7850
    Confusion Matrix:
[[11750 15400]
 [ 1150  4200]]


## SHAP Analysis

In [7]:
shap_results = compute_shap_values(xgb_results['model'], X_test, feature_names)

if shap_results['shap_values'] is not None:
    fig = create_shap_summary(shap_results['shap_values'], feature_names, shap_results['X_sample'])
    fig.show()


  ── Computing SHAP Values ──


    Top 10 SHAP features:
      unit_cost_usd: 0.2914
      disruption_score_current: 0.2702
      cusum_flag_rolling_4w: 0.2430
      demand_trend_slope: 0.2231
      current_inventory_weeks_of_cover: 0.1916
      demand_rolling_8w: 0.1693
      demand_rolling_4w: 0.1579
      sku_category_encoded: 0.1524
      clustering_coefficient: 0.1085
      demand_std_4w: 0.1069

    Feature Group Contributions:
      Forecasting: 35.0%
      Disruption Detection: 26.6%
      Graph: 22.4%
      Inventory: 16.1%


In [8]:
# Confusion matrix
if 'confusion_matrix' in ensemble:
    fig_cm = create_confusion_matrix_chart(ensemble['confusion_matrix'])
    fig_cm.show()

## Performance Summary

In [9]:
print('Model Performance Summary')
print('=' * 40)
print(f'XGBoost Val PR-AUC:   {xgb_results["val_prauc"]:.4f}')
print(f'LightGBM Val PR-AUC:  {lgb_results["val_prauc"]:.4f}')
print(f'Ensemble Val PR-AUC:  {ensemble["val_prauc"]:.4f}')
if 'test_prauc' in ensemble:
    print(f'Ensemble Test PR-AUC: {ensemble["test_prauc"]:.4f}')
    print(f'Precision@10%:        {ensemble.get("precision_at_k", 0):.4f}')

Model Performance Summary
XGBoost Val PR-AUC:   0.1181
LightGBM Val PR-AUC:  0.1194
Ensemble Val PR-AUC:  0.1193
Ensemble Test PR-AUC: 0.2933
Precision@10%:        0.3637
